# Bank Customer Churn: Cleaning, EDA & Prediction

Dataset: Kaggle **Bank Customer Churn Modelling** (`Churn_Modelling.csv`). Put the file next to this notebook.

**Pipeline:** clean → explore → engineer features → model (Logistic Regression vs Gradient Boosting) → explain → export risk list for the dashboard.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score,
                             roc_curve, precision_recall_curve, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})
RANDOM_STATE = 42

## 1. Load & clean

In [ ]:
df = pd.read_csv("Churn_Modelling.csv")
print(df.shape)
df.head()

In [ ]:
# Quality checks
print("Missing values:\n", df.isna().sum()[df.isna().sum() > 0])
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate CustomerId:", df["CustomerId"].duplicated().sum())
df.dtypes

In [ ]:
# Drop identifiers (no predictive value, and Surname would leak identity)
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"], errors="ignore")
df = df.drop_duplicates().reset_index(drop=True)

# Readable labels for EDA
df["Status"] = df["Exited"].map({0: "Retained", 1: "Churned"})
print(f"Churn rate: {df['Exited'].mean():.1%}  (imbalanced, so accuracy alone is misleading)")

## 2. Exploratory analysis

In [ ]:
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
for a, col in zip(ax.flat[:3], ["Geography", "Gender", "NumOfProducts"]):
    (df.groupby(col)["Exited"].mean() * 100).plot.bar(ax=a, color=sns.color_palette("viridis", 4), rot=0)
    a.set_title(f"Churn rate by {col}"); a.set_ylabel("%")
sns.kdeplot(data=df, x="Age", hue="Status", fill=True, ax=ax[1, 0], common_norm=False)
ax[1, 0].set_title("Age distribution")
sns.boxplot(data=df, x="Status", y="Balance", ax=ax[1, 1]); ax[1, 1].set_title("Balance vs churn")
(df.groupby("IsActiveMember")["Exited"].mean() * 100).plot.bar(ax=ax[1, 2], rot=0, color=["#ff6b81", "#00b8a9"])
ax[1, 2].set_title("Churn rate: inactive (0) vs active (1)")
plt.tight_layout(); plt.show()

In [ ]:
num = df.select_dtypes("number")
plt.figure(figsize=(9, 6))
sns.heatmap(num.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation matrix"); plt.show()

## 3. Feature engineering

In [ ]:
d = df.drop(columns="Status").copy()
d["ZeroBalance"]       = (d["Balance"] == 0).astype(int)
d["BalanceSalaryRatio"] = d["Balance"] / (d["EstimatedSalary"] + 1)
d["TenureByAge"]       = d["Tenure"] / d["Age"]
d["CreditScoreAge"]    = d["CreditScore"] / d["Age"]
d["MultiProduct"]      = (d["NumOfProducts"] >= 3).astype(int)   # 3-4 products churn heavily
d["Age2"]              = d["Age"] ** 2                           # churn peaks mid-life, not linear
d = pd.get_dummies(d, columns=["Geography", "Gender"], drop_first=True)

X, y = d.drop(columns="Exited"), d["Exited"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

scaler = StandardScaler().fit(X_train)   # fit on train only to avoid leakage
Xs_train, Xs_test = scaler.transform(X_train), scaler.transform(X_test)
X.shape

## 4. Models

In [ ]:
models = {
    "Logistic Regression": (LogisticRegression(max_iter=2000, class_weight="balanced"), True),
    "Gradient Boosting":   (GradientBoostingClassifier(random_state=RANDOM_STATE), False),
}
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
results, fitted = [], {}
for name, (m, scaled) in models.items():
    Xtr, Xte = (Xs_train, Xs_test) if scaled else (X_train, X_test)
    cv_auc = cross_val_score(m, Xtr, y_train, cv=cv, scoring="roc_auc").mean()
    m.fit(Xtr, y_train)
    proba = m.predict_proba(Xte)[:, 1]
    fitted[name] = (m, Xte, proba)
    results.append({"Model": name, "CV ROC-AUC": cv_auc, "Test ROC-AUC": roc_auc_score(y_test, proba),
                    "Test accuracy": (m.predict(Xte) == y_test).mean()})
pd.DataFrame(results).round(3)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
for name, (m, Xte, proba) in fitted.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax[0].plot(fpr, tpr, lw=2, label=f"{name} (AUC {roc_auc_score(y_test, proba):.3f})")
ax[0].plot([0, 1], [0, 1], "--", c="grey"); ax[0].set_title("ROC curves"); ax[0].legend()
ax[0].set_xlabel("False positive rate"); ax[0].set_ylabel("True positive rate")

best = max(fitted, key=lambda k: roc_auc_score(y_test, fitted[k][2]))
m, Xte, proba = fitted[best]
ConfusionMatrixDisplay.from_predictions(y_test, (proba >= .5).astype(int), ax=ax[1], cmap="Blues", colorbar=False)
ax[1].set_title(f"Confusion matrix: {best}"); plt.tight_layout(); plt.show()
print("Best model:", best)

### Choose a threshold that fits the business
Missing a churner costs more than calling a loyal customer, so tune the cut-off for **recall** instead of using 0.5.

In [ ]:
prec, rec, thr = precision_recall_curve(y_test, proba)
f2 = (5 * prec * rec) / (4 * prec + rec + 1e-9)          # F2 weights recall 2x
t = thr[np.argmax(f2[:-1])]
print(f"Chosen threshold: {t:.2f}")
print(classification_report(y_test, (proba >= t).astype(int), target_names=["Retained", "Churned"]))

## 5. What drives churn?

In [ ]:
pi = permutation_importance(m, Xte, y_test, scoring="roc_auc", n_repeats=10, random_state=RANDOM_STATE)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values().tail(12)
imp.plot.barh(figsize=(8, 6), color=sns.color_palette("viridis", 12))
plt.title("Permutation importance (drop in ROC-AUC)"); plt.tight_layout(); plt.show()

## 6. Export for the dashboard

In [ ]:
# Score every customer and bucket into risk tiers
all_proba = m.predict_proba(X if best == "Gradient Boosting" else scaler.transform(X))[:, 1]
out = df.drop(columns="Status").copy()
out["ChurnProbability"] = all_proba.round(4)
out["RiskTier"] = pd.cut(all_proba, [0, .3, .6, 1], labels=["Low", "Medium", "High"], include_lowest=True)

out.to_csv("Cleaned_bank_churn.csv", index=False)
out.sort_values("ChurnProbability", ascending=False).head(500).to_csv("top500_at_risk.csv", index=False)
print(out["RiskTier"].value_counts())

## Key takeaways (from a run on the real Kaggle file)
- 10,000 customers, no missing values or duplicates; **20.4% churn**, so judge models by ROC-AUC and recall, not accuracy.
- **Gradient Boosting** (test ROC-AUC 0.870) beat Logistic Regression (0.848).
- Strongest drivers by permutation importance: **number of products**, **active-member status**, **age**, **Germany**, and holding **3+ products**.
- With a tuned threshold (~0.16) the model catches about **82% of churners**, at ~46% precision.
- Predicted risk tiers line up with reality: actual churn was ~9% (Low), ~46% (Medium) and ~88% (High).
- Scores are computed on all customers including training rows, so tier rates are slightly optimistic.
